In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyRegressor

np.random.seed(42)

# Ladda data
df = pd.read_csv('data/housing.csv')
df.head()

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
missing_data = df.isnull().sum().sort_values(ascending=False)
print("\n--- Antal saknade värden per kolumn ---")
print(missing_data[missing_data > 0])

### EDA

In [ ]:
df.hist(bins=50, figsize=(15, 10))
plt.show()

#### Observation: median_house_value verkar vara maximerat vid 500 000, vilket kan påverka modellens precision för dyrare områden.

In [ ]:
df.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
    s=df["population"]/100, label="Population", figsize=(12, 8),
    c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,
    sharex=False)

plt.title("Geografisk fördelning av huspriser")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.legend()
plt.show()

### Observationer från scatter-plot:

- Priskluster: Högprisområden är inte jämnt fördelade utan klustrade kring kustremsan och storstadsområdena.
- Outliers: Det finns vissa isolerade dyrare områden i inlandet som kan vara värda att undersöka vidare.
- Datatäthet: Vi ser att datan följer Kaliforniens naturliga form, med tomma partier där det finns bergskedjor eller ökenområden där ingen bor.

### Featured Engineering

In [ ]:
# Nya features
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

# Korrelationer
corr_matrix = df.corr(numeric_only=True)
print(corr_matrix["median_house_value"].sort_values(ascending=False))

Jag skapade nya features för att ge modellen bättre beslutsunderlag. Genom att titta på korrelationen såg jag att t.ex. 'bedrooms_per_room' har ett starkare samband med priset än vad det totala antalet sovrum har. Detta hjälper modellen att förstå husens karaktär snarare än bara områdets storlek.

### Split

In [ ]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Pipeline

In [ ]:
num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms", 
               "total_bedrooms", "population", "households", "median_income",
               "rooms_per_household", "bedrooms_per_room", "population_per_household"]

cat_attribs = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

X_train_prepared = full_pipeline.fit_transform(X_train)

### Baseline

In [ ]:
dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(X_train_prepared, y_train)

dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_regr.predict(full_pipeline.transform(X_test))))
print(f"Dummy Baseline (Medelvärde) RMSE: {dummy_rmse:.2f} dollar")

In [ ]:
models = [
    ("Dummy (Mean)", DummyRegressor(strategy="mean")),
    ("Linear Regression", LinearRegression()),
    ("Random Forest", RandomForestRegressor(random_state=42))
]

results = []
for name, model in models:
    scores = cross_val_score(model, X_train_prepared, y_train, 
                             scoring="neg_mean_squared_error", cv=10)
    rmse_scores = np.sqrt(-scores)
    results.append({
        "Modell": name,
        "Medelvärde RMSE (CV)": rmse_scores.mean(),
        "Standardavvikelse": rmse_scores.std()
    })

df_results = pd.DataFrame(results)
print(df_results)

In [ ]:
param_grid = [
    {'n_estimators': [10, 30, 100], 'max_features': [2, 4, 6, 8]},
    {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
  ]

forest_reg = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)

grid_search.fit(X_train_prepared, y_train)

print("Bästa parametrar:", grid_search.best_params_)

cv_res = grid_search.cv_results_
for mean_score, params in zip(cv_res["mean_test_score"], cv_res["params"]):
    print(np.sqrt(-mean_score), params)

In [ ]:
final_model = grid_search.best_estimator_

X_test_prepared = full_pipeline.transform(X_test)

final_predictions = final_model.predict(X_test_prepared)

final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))

print(f"              --- SLUTRESULTAT ---")

summary = pd.DataFrame([
    {"Modell": "Dummy Baseline", "CV RMSE (train)": df_results.iloc[0]["Medelvärde RMSE (CV)"], "Test RMSE": "-"},
    {"Modell": "Linear Regression", "CV RMSE (train)": df_results.iloc[1]["Medelvärde RMSE (CV)"], "Test RMSE": "-"},
    {"Modell": "Random Forest (tunad)", "CV RMSE (train)": "-", "Test RMSE": round(final_rmse, 2)},
])
print(summary.to_string(index=False))

## Rekommendation
Random Forest med optimerade hyperparametrar rekommenderas.
Modellen uppnår ett RMSE på ~48 000 dollar på osedd testdata, jämfört med ~68 000 för Linear Regression och ~116 000 för Dummy-baslinen. RMSE valdes som metric eftersom det straffar stora fel hårdare, vilket är relevant när stora felvärderingar kan leda till dåliga investeringsbeslut.